# LeetCode 1291 — Sequential Digits bằng Python

Link đề: https://leetcode.com/problems/sequential-digits/ · Độ khó: **Medium** · Chủ đề: Enumeration

Skeleton LeetCode:

```python
class Solution:
    def sequentialDigits(self, low: int, high: int) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng **không gian đáp án rất nhỏ** (chỉ có 36 số có chữ số liên tiếp từ 2 chữ số trở lên) dù khoảng `[low, high]` có thể dài tới `10^9`.
2. Đi từ **vét cạn mọi số trong khoảng** (không khả thi) sang **sinh trực tiếp** các ứng viên.
3. Thành thạo ba kỹ thuật sinh: **cửa sổ trượt trên chuỗi `"123456789"`**, **xây số bằng phép toán** `x*10 + d`, và **BFS bằng hàng đợi** (tự nhiên cho ra thứ tự tăng).
4. Hiểu khi nào cần **sắp xếp** kết quả và cách tránh bằng thứ tự sinh hợp lý.
5. Tránh lỗi: biên `range` thiếu cửa sổ cuối, so sánh **chuỗi** thay vì **số**, nhầm "tăng dần" với "tăng đúng 1".

## 2. Đề bài & ví dụ

Một số nguyên có **chữ số liên tiếp** (sequential digits) khi và chỉ khi mỗi chữ số **lớn hơn chữ số đứng trước đúng 1** (ví dụ `1234`, `567`). Trả về **danh sách đã sắp xếp** mọi số trong đoạn `[low, high]` (bao gồm hai đầu) có chữ số liên tiếp.

**Ví dụ 1:** `low = 100, high = 300` → `[123, 234]`.

**Ví dụ 2:** `low = 1000, high = 13000` → `[1234, 2345, 3456, 4567, 5678, 6789, 12345]`.

**Ràng buộc:** `10 <= low <= high <= 10^9`.

In [1]:
import bisect
import random
import time
from collections import deque
from typing import List

examples = [(100, 300, [123, 234]), (1000, 13000, [1234, 2345, 3456, 4567, 5678, 6789, 12345])]
for low, high, want in examples:
    print(f"low={low}, high={high} -> mong đợi {want}")

low=100, high=300 -> mong đợi [123, 234]
low=1000, high=13000 -> mong đợi [1234, 2345, 3456, 4567, 5678, 6789, 12345]


## 3. Kiến thức nền

### 3.1 Chuyển qua lại giữa số và chuỗi

`str(1234)` → `"1234"`; `int("1234")` → `1234`; `len(str(x))` là số chữ số. Duyệt chuỗi cho từng chữ số dưới dạng ký tự; `int(ch)` đổi ra số.

In [2]:
x = 1234
s = str(x)
print(s, len(s), [int(ch) for ch in s], int("0789"))

1234 4 [1, 2, 3, 4] 789


### 3.2 Cắt chuỗi (slicing) và cửa sổ trượt

`s[i:i+L]` lấy đoạn dài `L` bắt đầu ở `i`. Với chuỗi dài `N`, các vị trí bắt đầu hợp lệ là `0 … N−L` ⇒ `range(N - L + 1)`.

In [3]:
digits = "123456789"
L = 4
print([digits[i:i + L] for i in range(len(digits) - L + 1)])

['1234', '2345', '3456', '4567', '5678', '6789']


### 3.3 Xây số bằng phép toán: `x * 10 + d`

Nối chữ số `d` vào cuối số `x`: `x * 10 + d`. Chữ số cuối của `x` là `x % 10`.

In [4]:
x = 12
while x % 10 < 9:
    x = x * 10 + (x % 10 + 1)
    print(x)

123
1234
12345
123456
1234567
12345678
123456789


### 3.4 Hàng đợi `deque` cho BFS

`deque.popleft()` lấy phần tử đầu trong `O(1)`. BFS duyệt "theo tầng": mọi số 1 chữ số, rồi mọi số 2 chữ số, … — trong mỗi tầng, thứ tự theo chữ số đầu.

In [5]:
q = deque([1, 2, 3])
q.append(12)
print(q.popleft(), q.popleft(), list(q))

1 2 [3, 12]


### 3.5 `bisect` để lọc nhanh trên danh sách đã sắp xếp

`bisect_left(a, low)` = vị trí đầu tiên `≥ low`; `bisect_right(a, high)` = vị trí đầu tiên `> high`. Đoạn giữa chính là các phần tử trong `[low, high]`.

In [6]:
a = [12, 23, 34, 123, 234, 345]
lo_i, hi_i = bisect.bisect_left(a, 30), bisect.bisect_right(a, 234)
print(a[lo_i:hi_i])

[34, 123, 234]


## 4. Ý tưởng nền tảng / trực giác

Một số có chữ số liên tiếp được **xác định hoàn toàn** bởi hai thứ: **chữ số đầu** `d` và **độ dài** `L` (ví dụ `d = 3, L = 4` ⇒ `3456`). Điều kiện: chữ số cuối `d + L − 1 ≤ 9`.

Đếm tay số ứng viên theo độ dài:

| L | các số | số lượng |
|---|---|---|
| 2 | 12, 23, …, 89 | 8 |
| 3 | 123, …, 789 | 7 |
| 4 | 1234, …, 6789 | 6 |
| … | … | … |
| 9 | 123456789 | 1 |

Tổng `8 + 7 + … + 1 = 36` số. Mọi ứng viên đều là một **đoạn con liên tục** của chuỗi `"123456789"`.

Vậy dù `high` lên tới `10^9`, đáp án có **tối đa 36 phần tử** ⇒ chỉ cần sinh 36 ứng viên và lọc.

In [7]:
all_seq = [int("123456789"[i:i + L]) for L in range(2, 10) for i in range(10 - L)]
print("tổng số ứng viên:", len(all_seq))
print(all_seq)

tổng số ứng viên: 36
[12, 23, 34, 45, 56, 67, 78, 89, 123, 234, 345, 456, 567, 678, 789, 1234, 2345, 3456, 4567, 5678, 6789, 12345, 23456, 34567, 45678, 56789, 123456, 234567, 345678, 456789, 1234567, 2345678, 3456789, 12345678, 23456789, 123456789]


## 5. Cách 1 — Brute Force / baseline: duyệt mọi số trong `[low, high]`

**Ý tưởng:** với mỗi `x` từ `low` đến `high`, kiểm tra các chữ số liên tiếp.

In [8]:
def is_sequential(x):
    s = str(x)
    return all(int(s[i + 1]) - int(s[i]) == 1 for i in range(len(s) - 1))

def sequential_brute(low, high):
    return [x for x in range(low, high + 1) if is_sequential(x)]

for low, high, want in examples:
    print(sequential_brute(low, high), "| mong đợi", want)

[123, 234] | mong đợi [123, 234]
[1234, 2345, 3456, 4567, 5678, 6789, 12345] | mong đợi [1234, 2345, 3456, 4567, 5678, 6789, 12345]


**Độ phức tạp:** `O((high − low) · D)` với `D ≤ 10` chữ số. Khoảng `[10, 10^9]` có ~`10^9` số ⇒ hàng chục phút trong Python. **Không khả thi.**

**Nút thắt:** đa số số trong khoảng **không** phải ứng viên (chỉ 36 số trong cả tỉ số). Ta đang "tìm kim đáy bể" thay vì **tự đúc cây kim**.

In [9]:
t = time.perf_counter()
sequential_brute(10, 10**6)
dt = time.perf_counter() - t
print(f"brute force [10, 10^6]: {dt:.2f}s -> ước lượng [10, 10^9]: ~{dt * 1000 / 60:.0f} phút")

brute force [10, 10^6]: 0.61s -> ước lượng [10, 10^9]: ~10 phút


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** thay vì kiểm tra, hãy **sinh** — mỗi ứng viên là `"123456789"[i : i+L]`.

**Thứ tự để khỏi sort:** duyệt `L` **tăng dần**, trong mỗi `L` duyệt `i` **tăng dần**. Số có nhiều chữ số hơn luôn lớn hơn; cùng số chữ số thì chữ số đầu lớn hơn ⇒ số lớn hơn. Vậy kết quả tự động tăng dần.

**Cắt tỉa theo độ dài:** chỉ cần `L` từ `len(str(low))` đến `len(str(high))`.

**Quy tắc "nếu… thì…":**
- Nếu ứng viên `< low` ⇒ bỏ qua, đi tiếp.
- Nếu ứng viên `> high` ⇒ mọi ứng viên sau (theo thứ tự tăng) cũng `> high` ⇒ **dừng hẳn**.
- Nếu `d + L − 1 > 9` ⇒ không tồn tại số bắt đầu bằng `d` dài `L`.

**Invariant:** danh sách kết quả luôn tăng dần và chỉ chứa số trong `[low, high]`.

## 7. Các cách tiếp theo

### Cách 2 — Cửa sổ trượt trên `"123456789"`

**Từng đoạn:**
1. `L` chạy từ `len(str(low))` đến `len(str(high))`.
2. `i` chạy `0 … 9 − L` (tức `range(10 − L)`).
3. `x = int(digits[i:i+L])`; nếu trong khoảng thì thêm; nếu `> high` thì trả kết quả luôn.

In [10]:
def sequential_window(low, high):
    digits = "123456789"
    res = []
    for L in range(len(str(low)), len(str(high)) + 1):
        for i in range(10 - L):
            x = int(digits[i:i + L])
            if x > high:
                return res               # mọi số sau còn lớn hơn
            if x >= low:
                res.append(x)
    return res

for low, high, want in examples:
    print(sequential_window(low, high), "| mong đợi", want)

[123, 234] | mong đợi [123, 234]
[1234, 2345, 3456, 4567, 5678, 6789, 12345] | mong đợi [1234, 2345, 3456, 4567, 5678, 6789, 12345]


In [11]:
class SolutionWindow:
    def sequentialDigits(self, low: int, high: int) -> List[int]:
        digits = "123456789"
        res = []
        for L in range(len(str(low)), len(str(high)) + 1):
            for i in range(10 - L):
                x = int(digits[i:i + L])
                if low <= x <= high:
                    res.append(x)
        return res

print([SolutionWindow().sequentialDigits(lo, hi) for lo, hi, _ in examples])

[[123, 234], [1234, 2345, 3456, 4567, 5678, 6789, 12345]]


**Độ phức tạp Cách 2:** tối đa 36 ứng viên, mỗi ứng viên `O(9)` ⇒ `O(1)` (hằng số). Bộ nhớ `O(1)` ngoài output.

### Cách 3 — Xây số bằng phép toán theo chữ số đầu, rồi sort

**Từng đoạn:**
1. Với mỗi chữ số đầu `d = 1 … 9`: `x = d`.
2. Lặp: nối chữ số kế `x = x * 10 + (x % 10 + 1)` khi chữ số cuối `< 9`.
3. Nếu `low ≤ x ≤ high` thì thêm. Nếu `x > high` thì dừng nhánh này.
4. Cuối cùng **sort** (vì thứ tự sinh là `12, 123, 1234, …, 23, 234, …` — không tăng).

In [12]:
def sequential_build(low, high):
    res = []
    for d in range(1, 10):
        x = d
        while x <= high and x % 10 < 9:
            x = x * 10 + (x % 10 + 1)
            if low <= x <= high:
                res.append(x)
    return sorted(res)

for low, high, want in examples:
    print(sequential_build(low, high), "| mong đợi", want)

[123, 234] | mong đợi [123, 234]
[1234, 2345, 3456, 4567, 5678, 6789, 12345] | mong đợi [1234, 2345, 3456, 4567, 5678, 6789, 12345]


**Độ phức tạp Cách 3:** `O(36 log 36) = O(1)`. Cách này không cần chuyển chuỗi, gần với tư duy "DFS từ chữ số đầu" của gợi ý đề.

### Cách 4 — BFS bằng hàng đợi (tự động tăng dần)

**Ý tưởng:** bắt đầu hàng đợi với `1 … 9`. Mỗi lần lấy `x` ra: nếu trong khoảng thì thêm vào kết quả; nếu chữ số cuối `< 9` thì đẩy `x * 10 + (x%10 + 1)` vào cuối. Vì BFS đi theo tầng (độ dài) và trong mỗi tầng giữ thứ tự chữ số đầu, kết quả **đã sắp xếp**.

In [13]:
def sequential_bfs(low, high):
    res = []
    q = deque(range(1, 10))
    while q:
        x = q.popleft()
        if x > high:
            continue                     # các số con của x còn lớn hơn -> bỏ
        if x >= low:
            res.append(x)
        last = x % 10
        if last < 9:
            q.append(x * 10 + last + 1)
    return res

for low, high, want in examples:
    print(sequential_bfs(low, high), "| mong đợi", want)

[123, 234] | mong đợi [123, 234]
[1234, 2345, 3456, 4567, 5678, 6789, 12345] | mong đợi [1234, 2345, 3456, 4567, 5678, 6789, 12345]


In [14]:
class SolutionBFS:
    def sequentialDigits(self, low: int, high: int) -> List[int]:
        res = []
        q = deque(range(1, 10))
        while q:
            x = q.popleft()
            if low <= x <= high:
                res.append(x)
            if x % 10 < 9 and x * 10 < high:     # nhánh con có thể còn <= high
                q.append(x * 10 + x % 10 + 1)
        return res

print([SolutionBFS().sequentialDigits(lo, hi) for lo, hi, _ in examples])

[[123, 234], [1234, 2345, 3456, 4567, 5678, 6789, 12345]]


**Độ phức tạp Cách 4:** tối đa `9 + 36` lần đẩy/lấy ⇒ `O(1)`.

### Cách 5 — Tính sẵn 36 số một lần, lọc bằng `bisect`

Nếu hàm được gọi **rất nhiều lần** (ví dụ hàng triệu truy vấn), tính sẵn danh sách 36 số đã sắp xếp ở mức module, mỗi truy vấn chỉ cần hai lần `bisect` ⇒ `O(log 36)`.

In [15]:
ALL_SEQ = sorted(int("123456789"[i:i + L]) for L in range(2, 10) for i in range(10 - L))

def sequential_precomputed(low, high):
    return ALL_SEQ[bisect.bisect_left(ALL_SEQ, low): bisect.bisect_right(ALL_SEQ, high)]

for low, high, want in examples:
    print(sequential_precomputed(low, high), "| mong đợi", want)

[123, 234] | mong đợi [123, 234]
[1234, 2345, 3456, 4567, 5678, 6789, 12345] | mong đợi [1234, 2345, 3456, 4567, 5678, 6789, 12345]


## 8. TRACE từng bước

Trace Cách 2 với `low = 1000, high = 13000`: `L` chạy từ 4 đến 5.

| L | i | cửa sổ | x | so với [1000, 13000] | hành động |
|---|---|---|---|---|---|
| 4 | 0 | 1234 | 1234 | trong | thêm |
| 4 | 1..5 | 2345 … 6789 | | trong | thêm |
| 5 | 0 | 12345 | 12345 | trong | thêm |
| 5 | 1 | 23456 | 23456 | > high | **dừng** |

In [16]:
def trace_window(low, high):
    digits = "123456789"
    res = []
    for L in range(len(str(low)), len(str(high)) + 1):
        for i in range(10 - L):
            x = int(digits[i:i + L])
            if x > high:
                print(f"L={L}, i={i}: x={x} > high={high} -> DỪNG")
                return res
            status = "thêm" if x >= low else "< low, bỏ"
            if x >= low:
                res.append(x)
            print(f"L={L}, i={i}: cửa sổ '{digits[i:i + L]}' -> x={x}, {status}, res={res}")
    return res

print("đáp án =", trace_window(1000, 13000))

L=4, i=0: cửa sổ '1234' -> x=1234, thêm, res=[1234]
L=4, i=1: cửa sổ '2345' -> x=2345, thêm, res=[1234, 2345]
L=4, i=2: cửa sổ '3456' -> x=3456, thêm, res=[1234, 2345, 3456]
L=4, i=3: cửa sổ '4567' -> x=4567, thêm, res=[1234, 2345, 3456, 4567]
L=4, i=4: cửa sổ '5678' -> x=5678, thêm, res=[1234, 2345, 3456, 4567, 5678]
L=4, i=5: cửa sổ '6789' -> x=6789, thêm, res=[1234, 2345, 3456, 4567, 5678, 6789]
L=5, i=0: cửa sổ '12345' -> x=12345, thêm, res=[1234, 2345, 3456, 4567, 5678, 6789, 12345]
L=5, i=1: x=23456 > high=13000 -> DỪNG
đáp án = [1234, 2345, 3456, 4567, 5678, 6789, 12345]


In [17]:
def trace_bfs(low, high):
    res = []
    q = deque(range(1, 10))
    step = 0
    while q:
        x = q.popleft()
        step += 1
        note = ""
        if x > high:
            note = "> high, bỏ cả nhánh"
        else:
            if x >= low:
                res.append(x)
                note = "thêm"
            if x % 10 < 9:
                q.append(x * 10 + x % 10 + 1)
                note += f" | đẩy {q[-1]}"
        if step <= 12 or note.startswith("thêm"):
            print(f"lấy {x:>4}: {note:<26} hàng đợi còn {len(q)}")
    return res

print("đáp án =", trace_bfs(100, 300))

lấy    1:  | đẩy 12                  hàng đợi còn 9
lấy    2:  | đẩy 23                  hàng đợi còn 9
lấy    3:  | đẩy 34                  hàng đợi còn 9
lấy    4:  | đẩy 45                  hàng đợi còn 9
lấy    5:  | đẩy 56                  hàng đợi còn 9
lấy    6:  | đẩy 67                  hàng đợi còn 9
lấy    7:  | đẩy 78                  hàng đợi còn 9
lấy    8:  | đẩy 89                  hàng đợi còn 9
lấy    9:                            hàng đợi còn 8
lấy   12:  | đẩy 123                 hàng đợi còn 8
lấy   23:  | đẩy 234                 hàng đợi còn 8
lấy   34:  | đẩy 345                 hàng đợi còn 8
lấy  123: thêm | đẩy 1234            hàng đợi còn 7
lấy  234: thêm | đẩy 2345            hàng đợi còn 7
đáp án = [123, 234]


## 9. Code nộp LeetCode

Chọn Cách 2 (cửa sổ trượt + dừng sớm): ngắn, kết quả tự sắp xếp, không cần sort.

In [18]:
class Solution:
    def sequentialDigits(self, low: int, high: int) -> List[int]:
        digits = "123456789"
        res = []
        for length in range(len(str(low)), len(str(high)) + 1):
            for start in range(10 - length):          # 10 - length cửa sổ
                x = int(digits[start:start + length])
                if x > high:
                    return res                        # các số sau còn lớn hơn
                if x >= low:
                    res.append(x)
        return res

for low, high, want in examples:
    got = Solution().sequentialDigits(low, high)
    print(got, "OK" if got == want else "SAI")

[123, 234] OK
[1234, 2345, 3456, 4567, 5678, 6789, 12345] OK


## 10. Vì sao thuật toán đúng

1. **Đủ:** số `x` có chữ số liên tiếp, dài `L`, chữ số đầu `d` ⇒ các chữ số là `d, d+1, …, d+L−1`, đúng bằng `"123456789"[d−1 : d−1+L]` ⇒ `x` được sinh ra (với `start = d − 1 ≤ 9 − L`). Vì `low ≤ x ≤ high` nên `len(str(low)) ≤ L ≤ len(str(high))`, tức `L` nằm trong vòng lặp.
2. **Không thừa:** mọi cửa sổ của `"123456789"` đều có chữ số tăng đúng 1 ⇒ là số hợp lệ; ta chỉ thêm khi nằm trong `[low, high]`.
3. **Thứ tự tăng:** với hai ứng viên `(L1, s1)` đi trước `(L2, s2)`: hoặc `L1 < L2` (ít chữ số hơn ⇒ nhỏ hơn), hoặc `L1 = L2, s1 < s2` (cùng độ dài, chữ số đầu nhỏ hơn ⇒ nhỏ hơn).
4. **Dừng sớm đúng:** do (3), khi gặp `x > high` thì mọi ứng viên sau cũng `> high`.

## 11. Phân tích Time / Space

Gọi `D = 9` (số chữ số tối đa có thể có trong số liên tiếp).

| Cách | Time | Space (ngoài output) |
|---|---|---|
| 1. Brute force | `O((high − low)·D)` — tới `10^10` | `O(1)` |
| 2. Cửa sổ trượt | `O(D²)` = `O(1)` (≤ 36 ứng viên) | `O(1)` |
| 3. Xây số + sort | `O(D² log D)` = `O(1)` | `O(1)` |
| 4. BFS | `O(D²)` = `O(1)` | `O(D)` hàng đợi |
| 5. Tính sẵn + bisect | `O(log 36)` mỗi truy vấn | `O(36)` |

Output có tối đa 36 phần tử.

## 12. Bộ test

Ví dụ đề; edge case (`low = high`, khoảng không chứa số nào, `[10, 10^9]` chứa đủ 36 số, `high = 123456789`, `low` vượt mọi ứng viên); đối chiếu với brute force trên nhiều khoảng nhỏ ngẫu nhiên (seed cố định) và đối chiếu giữa các cách trên khoảng lớn ngẫu nhiên.

In [19]:
ALL = {
    "window": sequential_window, "build": sequential_build, "bfs": sequential_bfs,
    "precomputed": sequential_precomputed,
    "SolWindow": lambda a, b: SolutionWindow().sequentialDigits(a, b),
    "SolBFS": lambda a, b: SolutionBFS().sequentialDigits(a, b),
    "Solution": lambda a, b: Solution().sequentialDigits(a, b),
}
fixed = [
    (100, 300, [123, 234]),
    (1000, 13000, [1234, 2345, 3456, 4567, 5678, 6789, 12345]),
    (10, 10, []), (12, 12, [12]), (89, 89, [89]),
    (90, 122, []),                                  # khoảng trống giữa 89 và 123
    (10, 10**9, ALL_SEQ),                           # cả 36 số
    (123456789, 10**9, [123456789]),
    (123456790, 10**9, []),
    (58, 155, [67, 78, 89, 123]),
]
for low, high, want in fixed:
    for name, f in ALL.items():
        got = f(low, high)
        assert got == want, (name, low, high, got, want)
print("Test cố định: qua", len(fixed), "case; tổng số ứng viên =", len(ALL_SEQ))

Test cố định: qua 10 case; tổng số ứng viên = 36


In [20]:
rng = random.Random(1291)
for _ in range(300):
    low = rng.randint(10, 20000)
    high = rng.randint(low, min(low + rng.choice([50, 500, 5000]), 30000))
    want = sequential_brute(low, high)
    for name, f in ALL.items():
        assert f(low, high) == want, (name, low, high)
for _ in range(3000):
    a, b = sorted(rng.randint(10, 10**9) for _ in range(2))
    ref = sequential_precomputed(a, b)
    assert ref == sorted(ref) and all(a <= x <= b and is_sequential(x) for x in ref)
    for name, f in ALL.items():
        assert f(a, b) == ref, (name, a, b)
print("300 khoảng nhỏ (so brute force) + 3000 khoảng lớn: mọi cách khớp nhau")

300 khoảng nhỏ (so brute force) + 3000 khoảng lớn: mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Biên `range(9 - L)` thiếu cửa sổ cuối

In [21]:
def wrong_range(low, high):
    digits = "123456789"
    res = []
    for L in range(len(str(low)), len(str(high)) + 1):
        for i in range(9 - L):                       # SAI: thiếu i = 9 - L
            x = int(digits[i:i + L])
            if low <= x <= high:
                res.append(x)
    return res

print("Code sai:", wrong_range(1000, 13000))
print("Đúng là :", Solution().sequentialDigits(1000, 13000))

Code sai: [1234, 2345, 3456, 4567, 5678, 12345]
Đúng là : [1234, 2345, 3456, 4567, 5678, 6789, 12345]


Thiếu `6789` (cửa sổ cuối kết thúc ở chữ số 9). Chuỗi dài 9, cửa sổ dài `L` có `9 − L + 1 = 10 − L` vị trí. **Code đúng:** `range(10 - L)`.

### Lỗi 2 — Sinh theo chữ số đầu nhưng quên sort

In [22]:
def wrong_no_sort(low, high):
    res = []
    for d in range(1, 10):
        x = d
        while x % 10 < 9:
            x = x * 10 + x % 10 + 1
            if low <= x <= high:
                res.append(x)
    return res                                        # SAI: chưa sắp xếp

got = wrong_no_sort(10, 1000)
print("Code sai:", got)
print("đã tăng dần?", got == sorted(got), "| đúng là:", Solution().sequentialDigits(10, 1000))

Code sai: [12, 123, 23, 234, 34, 345, 45, 456, 56, 567, 67, 678, 78, 789, 89]
đã tăng dần? False | đúng là: [12, 23, 34, 45, 56, 67, 78, 89, 123, 234, 345, 456, 567, 678, 789]


Thứ tự sinh là `12, 123, 23, 234, …` — không tăng. **Code đúng:** `return sorted(res)` (Cách 3), hoặc sinh theo độ dài trước (Cách 2/4).

### Lỗi 3 — So sánh **chuỗi** thay vì **số**

In [23]:
def wrong_string_compare(low, high):
    digits = "123456789"
    res = []
    for L in range(2, 10):
        for i in range(10 - L):
            s = digits[i:i + L]
            if str(low) <= s <= str(high):            # SAI: so sánh từ điển
                res.append(int(s))
    return sorted(res)

print("Code sai:", wrong_string_compare(100, 300))
print("Đúng là :", Solution().sequentialDigits(100, 300))

Code sai: [12, 23, 123, 234, 1234, 2345, 12345, 23456, 123456, 234567, 1234567, 2345678, 12345678, 23456789, 123456789]
Đúng là : [123, 234]


Theo thứ tự từ điển `"100" ≤ "1234" ≤ "300"` (vì so ký tự đầu `'1' < '3'`), nên số 1234 bị đưa vào nhầm. **Code đúng:** đổi sang `int` rồi mới so sánh.

### Lỗi 4 — Kiểm tra "chữ số tăng dần" thay vì "tăng đúng 1"

In [24]:
def wrong_increasing(x):
    s = str(x)
    return all(s[i] < s[i + 1] for i in range(len(s) - 1))     # SAI

def wrong_brute(low, high):
    return [x for x in range(low, high + 1) if wrong_increasing(x)]

print("Code sai:", wrong_brute(120, 140))
print("Đúng là :", sequential_brute(120, 140))

Code sai: [123, 124, 125, 126, 127, 128, 129, 134, 135, 136, 137, 138, 139]
Đúng là : [123]


`124`, `125`, … có chữ số tăng nhưng không liên tiếp. **Code đúng:** `int(s[i+1]) - int(s[i]) == 1`.

### Lỗi 5 — BFS dừng hẳn khi gặp số `> high` (dùng `break` thay vì bỏ qua)

Trong BFS, gặp một số `> high` không có nghĩa là mọi số **còn trong hàng đợi** đều lớn hơn — nếu hàng đợi chưa được sắp xếp theo đúng thứ tự (ví dụ khởi tạo sai thứ tự), `break` sẽ bỏ sót.

In [25]:
def wrong_bfs_break(low, high):
    res = []
    q = deque([9, 8, 7, 6, 5, 4, 3, 2, 1])           # khởi tạo ngược thứ tự
    while q:
        x = q.popleft()
        if x > high:
            break                                   # SAI với thứ tự này
        if x >= low:
            res.append(x)
        if x % 10 < 9:
            q.append(x * 10 + x % 10 + 1)
    return sorted(res)

print("Code sai:", wrong_bfs_break(10, 50))
print("Đúng là :", Solution().sequentialDigits(10, 50))

Code sai: []
Đúng là : [12, 23, 34, 45]


**Code đúng:** khởi tạo `deque(range(1, 10))` (tăng dần), và dùng `continue` thay cho `break` để an toàn (Cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Khoảng lớn (`10^9`) nhưng tính chất rất "khắt khe" ⇒ **đếm số đáp án có thể có** trước; nếu nhỏ thì **sinh trực tiếp**.
- Số được xác định bởi vài tham số (chữ số đầu, độ dài) ⇒ duyệt các tham số thay vì duyệt số.
- Muốn kết quả tăng dần mà không sort ⇒ sinh theo **độ dài** rồi theo **chữ số đầu** (hoặc BFS).
- Luôn so sánh số dưới dạng `int`.
- Đếm biên cửa sổ: chuỗi dài `N`, cửa sổ dài `L` ⇒ `N − L + 1` vị trí.

## 15. Biến thể của bài

1. **Chữ số giảm liên tiếp** (`9876`, `543`): sinh từ chuỗi `"987654321"`.
2. **Bước nhảy `step`**: mỗi chữ số lớn hơn chữ số trước đúng `step` (Bài tập 5).
3. **Đếm** số có chữ số liên tiếp trong khoảng thay vì liệt kê.
4. **LC 967 Numbers With Same Consecutive Differences:** chênh lệch tuyệt đối giữa hai chữ số kề là `k` — BFS/DFS sinh số.
5. **LC 2081 / LC 906 (số đối xứng):** cũng là "sinh ứng viên thay vì duyệt khoảng".
6. **Stepping Numbers (LC 1215, Premium):** chữ số kề chênh nhau đúng 1 (tăng hoặc giảm) — BFS từ 0..9.

In [26]:
def stepping_numbers(low, high):           # chữ số kề chênh nhau 1 (tăng hoặc giảm)
    res = [0] if low == 0 else []
    q = deque(range(1, 10))
    while q:
        x = q.popleft()
        if x > high:
            continue
        if x >= low:
            res.append(x)
        last = x % 10
        for d in (last - 1, last + 1):
            if 0 <= d <= 9:
                q.append(x * 10 + d)
    return res

print(stepping_numbers(0, 21))
print(stepping_numbers(10, 15))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 21]
[10, 12]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | kiểm tra mọi số trong khoảng | `O((high−low)·D)` | `O(1)` | kiểm chứng khoảng nhỏ |
| 2. Cửa sổ trượt | đoạn con của `"123456789"` theo độ dài | `O(1)` | `O(1)` | **nộp bài** |
| 3. Xây số + sort | `x*10 + (x%10+1)` từ mỗi chữ số đầu | `O(1)` | `O(1)` | không muốn dùng chuỗi |
| 4. BFS | hàng đợi, tự sắp xếp | `O(1)` | `O(D)` | khi mở rộng sang quy tắc phức tạp hơn |
| 5. Tính sẵn + bisect | 36 số sẵn, lọc bằng nhị phân | `O(log 36)` | `O(36)` | nhiều truy vấn |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `sequentialDigits(10, 100)` và `sequentialDigits(58, 155)` trả về gì?

**Bài 2 (tính tay):** Có bao nhiêu số có chữ số liên tiếp **đúng 4 chữ số**? Trong `[10, 10^9]` có bao nhiêu số tất cả? Số nhỏ nhất có 6 chữ số là gì?

**Bài 3 (điền TODO):** Hoàn thiện `is_sequential_todo(x)` **không dùng chuỗi** (chỉ dùng `%` và `//`).

**Bài 4 (tự viết lại):** Tự viết lại lời giải BFS `sequential_bfs_mine(low, high)` (không nhìn mục 7).

**Bài 5 (mini challenge):** Viết `step_digits(low, high, step)` trả về các số trong `[low, high]` (≥ 2 chữ số) mà mỗi chữ số lớn hơn chữ số trước đúng `step` (`1 ≤ step ≤ 8`), đã sắp xếp.

In [27]:
def is_sequential_todo(x):
    # TODO 1: lấy chữ số cuối: last = x % 10, rồi x //= 10
    # TODO 2: lặp khi x > 0: chữ số kế (bên trái) phải bằng last - 1
    # TODO 3: trả True nếu mọi cặp đều thoả
    pass

def step_digits_todo(low, high, step):
    # TODO: với mỗi chữ số đầu d, nối d+step, d+2*step, ... khi còn <= 9
    pass

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Khoảng `[10, 100]` chỉ chứa số 2 chữ số. (2) `12, 23, …, 89`. (3) Với `[58, 155]`: số 2 chữ số `≥ 58` là `67, 78, 89`; số 3 chữ số `≤ 155` là `123`.

**Bài 2.** (1) Độ dài `L` có `10 − L` số. (2) `L = 4` ⇒ 6; tổng `Σ_{L=2}^{9} (10 − L) = 36`. (3) Số nhỏ nhất dài 6 là `123456`.

**Bài 3.** (1) Đọc chữ số từ phải sang trái. (2) Mỗi bước `d = x % 10` phải bằng `last − 1`. (3) Nhớ trường hợp số 1 chữ số (trả `True` theo định nghĩa trống, nhưng bài chỉ hỏi `≥ 10`).

**Bài 4.** (1) `deque(range(1, 10))`. (2) Lấy ra, kiểm tra khoảng, đẩy con `x*10 + last + 1` khi `last < 9`. (3) Dùng `continue` khi `x > high`.

**Bài 5.** (1) Giống Cách 3 với `+ step` thay cho `+ 1`. (2) Chỉ nối khi `last + step ≤ 9`. (3) Sort kết quả và đối chiếu với vét cạn trên khoảng nhỏ.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert Solution().sequentialDigits(10, 100) == [12, 23, 34, 45, 56, 67, 78, 89]
assert Solution().sequentialDigits(58, 155) == [67, 78, 89, 123]

# Bài 2
assert sum(1 for x in ALL_SEQ if len(str(x)) == 4) == 6
assert len(Solution().sequentialDigits(10, 10**9)) == 36
assert Solution().sequentialDigits(100000, 999999)[0] == 123456

# Bài 3
def is_sequential_sol(x):
    last = x % 10
    x //= 10
    while x > 0:
        d = x % 10
        if d != last - 1:
            return False
        last = d
        x //= 10
    return True

# Bài 4
def sequential_bfs_sol(low, high):
    res, q = [], deque(range(1, 10))
    while q:
        x = q.popleft()
        if x > high:
            continue
        if x >= low:
            res.append(x)
        if x % 10 < 9:
            q.append(x * 10 + x % 10 + 1)
    return res

# Bài 5
def step_digits_sol(low, high, step):
    res = []
    for d in range(1, 10):
        x = d
        while x % 10 + step <= 9:
            x = x * 10 + x % 10 + step
            if x > high:
                break
            if x >= low:
                res.append(x)
    return sorted(res)

def step_brute(low, high, step):
    out = []
    for x in range(max(low, 10), high + 1):
        s = str(x)
        if all(int(s[i + 1]) - int(s[i]) == step for i in range(len(s) - 1)):
            out.append(x)
    return out

for x in range(10, 50000):
    assert is_sequential_sol(x) == is_sequential(x)
rng = random.Random(9)
for _ in range(300):
    a = rng.randint(10, 20000); b = rng.randint(a, 30000)
    assert sequential_bfs_sol(a, b) == sequential_brute(a, b)
    st = rng.randint(1, 8)
    assert step_digits_sol(a, b, st) == step_brute(a, b, st)
assert step_digits_sol(10, 10**9, 1) == ALL_SEQ
assert step_digits_sol(10, 10**9, 2) == [13, 24, 35, 46, 57, 68, 79, 135, 246, 357, 468, 579, 1357, 2468, 3579, 13579]
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sinh ứng viên thay vì duyệt khoảng":**

```python
digits = "123456789"
res = []
for L in range(len(str(low)), len(str(high)) + 1):
    for i in range(10 - L):            # 10 - L cửa sổ
        x = int(digits[i:i + L])
        if x > high: return res        # dừng sớm (kết quả đang tăng)
        if x >= low: res.append(x)
return res
```

**Template BFS sinh số:**

```python
q = deque(range(1, 10))
while q:
    x = q.popleft()
    if x > high: continue
    if x >= low: res.append(x)
    for d in next_digits(x % 10):
        q.append(x * 10 + d)
```

**Checklist trước khi submit:**
- [ ] `range(10 - L)`, không phải `range(9 - L)`.
- [ ] So sánh bằng `int`, không bằng chuỗi.
- [ ] Kết quả **đã sắp xếp** (sinh theo độ dài hoặc `sorted`).
- [ ] Bao gồm cả hai đầu `low`, `high`.
- [ ] Khoảng không chứa số nào ⇒ trả `[]`.
- [ ] Đúng chữ ký: `sequentialDigits(self, low: int, high: int) -> List[int]`.